# PyTorch training loop over Zarr cutouts

**Environment:** [uv](https://docs.astral.sh/uv/) project venv from the [Quick-start](../docs/quickstart.md) (`uv venv --python 3.12 .venv`, `uv sync --extra dev`). In Jupyter, select kernel **Python (data-lake)** (`uv run python -m ipykernel install --user --name=data-lake --display-name "Python (data-lake)"`). See [notebooks.md](../docs/notebooks.md) for which notebooks need a real lake.

Demonstrates:
1. `CutoutDataset` — map-style; resolves every `_source_id` to a tile position **once at construction** (16 bytes per sample), then DataLoader workers read Zarr directly
2. `TileSequentialSampler` — yields samples grouped by HEALPix tile so consecutive reads hit the same shard
3. `TileCutoutDataset` — iterable streaming of whole tiles for a full-epoch pass

Needs a lake with a catalog and matching cutouts (defaults assume `des_dr2` at `/data/lake`; edit the next cell).

Tip: `dl-demo-lake /tmp/demo && export DATA_LAKE_ROOT=/tmp/demo/data` gives a tiny `DEMO_PHOT` cutout survey for a dry run.


In [ ]:
from pathlib import Path
import os

import numpy as np
import torch
from torch.utils.data import DataLoader

from data_lake.io.catalog import CatalogAccessor
from data_lake.ml.dataset import CutoutDataset, TileCutoutDataset, TileSequentialSampler

# Prefer $DATA_LAKE_ROOT, else a common shared path. Edit SURVEY to match your lake.
LAKE_ROOT = Path(os.environ.get("DATA_LAKE_ROOT", "/data/lake"))
SURVEY = "des_dr2"  # or "DEMO_PHOT" after dl-demo-lake
BATCH_SIZE = 64
NUM_WORKERS = 0  # set to 4 once the loop works; scripts with workers need if __name__ == "__main__"


## 1. Fetch source IDs to train on

Select objects that already have a cutout (`_cutout_index >= 0`). Column names differ by survey — check with `dl-describe-survey` or `cat.columns`. The lake join key is always `_source_id`.


In [ ]:
cat = CatalogAccessor(LAKE_ROOT, SURVEY)
print("Survey columns (first 20):", cat.columns[:20])

# Adjust the WHERE clause to columns that exist in your survey.
# Fallback: every source that has a cutout.
sid = cat.link_id_column  # usually "_source_id"
try:
    df = cat.query(f"""
        SELECT {sid} AS _source_id
        FROM catalog
        WHERE mag_i BETWEEN 18.0 AND 23.0
          AND _cutout_index >= 0
        LIMIT 5000
    """)
except Exception as exc:
    print(f"Science cut failed ({exc}); taking any sources with cutouts.")
    df = cat.query(f"""
        SELECT {sid} AS _source_id
        FROM catalog
        WHERE _cutout_index >= 0
        LIMIT 5000
    """)

source_ids = df["_source_id"].to_list()
print(f"Training set size: {len(source_ids):,}")


## 2. Map-style dataset with tile-grouped sampler

`CutoutDataset(..., catalog_accessor=cat)` runs **one** catalog scan when the dataset is built, then drops the accessor so DataLoader workers never share its DuckDB connection. Missing IDs are warned once; drop them with `ds.tile_npix >= 0` before training.

Prefer `TileSequentialSampler` over a random shuffle of all indices: it keeps consecutive batches inside the same HEALPix tile.


In [ ]:
from data_lake.io.cutouts import CutoutAccessor

def normalise(image: np.ndarray) -> torch.Tensor:
    """Log-scale stretch + normalise each band independently."""
    image = np.log1p(np.clip(image, 0, None))
    mean = image.mean(axis=(1, 2), keepdims=True)
    std = image.std(axis=(1, 2), keepdims=True) + 1e-6
    return torch.from_numpy(((image - mean) / std).astype(np.float32))

# Drop IDs with no cutout before building the dataset
with CutoutAccessor(LAKE_ROOT, SURVEY, catalog_accessor=cat) as acc:
    npix, _local = acc.locate(source_ids)
n_missing = int((npix < 0).sum())
if n_missing:
    print(f"Dropping {n_missing} ID(s) with no cutout")
    source_ids = [sid for sid, p in zip(source_ids, npix.tolist()) if p >= 0]

ds = CutoutDataset(
    lake_root=LAKE_ROOT,
    survey=SURVEY,
    source_ids=source_ids,
    transform=normalise,
    tile_sort=True,
    catalog_accessor=cat,
)
cat.close()  # dataset no longer needs it

print(ds)
print("Sample image shape:", ds[0].shape)
print("Unique tiles in sample:", len(np.unique(ds.tile_npix)))


In [ ]:
sampler = TileSequentialSampler(ds, shuffle_tiles=True, shuffle_within_tile=False)

loader = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)
print(f"Batches per epoch: {len(loader):,}")


## 3. Minimal training loop

In [ ]:
import torch.nn as nn

# Toy CNN – replace with your model
n_bands = ds[0].shape[0]
model = nn.Sequential(
    nn.Conv2d(n_bands, 32, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(8),
    nn.Flatten(),
    nn.Linear(32 * 8 * 8, 128), nn.ReLU(),
    nn.Linear(128, 1),
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

N_EPOCHS = 2  # increase for real training

for epoch in range(N_EPOCHS):
    running_loss = 0.0
    for step, batch in enumerate(loader):
        images = batch.to(device)
        # Dummy target: mean pixel value (replace with real labels)
        targets = images.mean(dim=(1, 2, 3), keepdim=False).unsqueeze(1)

        optimiser.zero_grad()
        preds = model(images)
        loss = loss_fn(preds, targets)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
        if step % 50 == 0:
            print(f"Epoch {epoch+1}  step {step:>5d}  loss={running_loss/(step+1):.4f}")

    print(f"Epoch {epoch+1} done, avg loss={running_loss/len(loader):.4f}")

## 4. Iterable dataset for full-epoch streaming

`TileCutoutDataset` walks whole HEALPix tiles (one worker owns each tile). Use this when you visit every cutout once per epoch and do not need random access by ID.


In [ ]:
tile_ds = TileCutoutDataset(
    lake_root=LAKE_ROOT,
    survey=SURVEY,
    transform=normalise,
    shuffle_tiles=True,   # shuffle tile order each epoch
)

tile_loader = DataLoader(
    tile_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

for step, batch in enumerate(tile_loader):
    print(f"Step {step}: batch shape {batch.shape}")
    if step >= 4:
        break